# LLM Foundations to Agents — Intensive 3-Hour Lab
**Target Hardware:** NVIDIA T4 GPU (16GB VRAM)  
**Difficulty:** Beginner–Intermediate (CS background assumed)  
**Duration:** ~3 hours | 25 Tasks across 5 Sections

---
**Sections:**
1. Foundations (45 min)
2. Quantization (30 min)
3. Inference & Prompting (30 min)
4. RAG with FAISS (45 min)
5. Agents (30 min)

In [ ]:
# @title ⚙️ Setup — Install All Dependencies (Run First)
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community
!pip install -q faiss-gpu sentence-transformers
!pip install -q ctransformers huggingface_hub
!pip install -q torch torchvision
!pip install -q datasets
print("✅ All packages installed.")

In [ ]:
# @title 🔧 Runtime Check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Go to Runtime > Change runtime type > T4 GPU.")

---
## Section 1: Foundations
**45 minutes | Tasks 1–5**

Core building blocks: tokenization via BPE, attention mechanics, and MoE routing.

In [ ]:
# @title Task 1 — BPE: Byte-Pair Encoding Statistics
# BPE builds a vocabulary by iteratively merging the most frequent symbol pair.
# Step 1: Count how often each ADJACENT pair of symbols appears across all words.

from collections import defaultdict

def get_vocab(corpus: list[str]) -> dict:
    """Convert corpus words into character sequences with end-of-word marker."""
    vocab = defaultdict(int)
    for word in corpus:
        # Represent each word as space-separated characters + </w> end marker
        vocab[' '.join(list(word)) + ' </w>'] += 1
    return dict(vocab)

def get_stats(vocab: dict) -> dict:
    """
    Count frequency of every adjacent symbol pair across all vocab entries.
    
    Args:
        vocab: dict mapping symbol-sequence strings to their corpus frequency.
    Returns:
        pairs: dict mapping (sym_a, sym_b) tuples to total co-occurrence count.
    """
    pairs = defaultdict(int)
    
    # --- MISSING CODE ---
    # Iterate over vocab items.
    # Split each key string into a list of symbols.
    # Count every adjacent (symbols[i], symbols[i+1]) pair, weighted by word_freq.
    # Store counts in `pairs`.
    # Hint: for word, freq in vocab.items(): symbols = word.split()
    
    return dict(pairs)

# Test
corpus = ["low", "lower", "newest", "widest", "low", "low"]
vocab = get_vocab(corpus)
print("Vocab:", vocab)
stats = get_stats(vocab)
print("\nTop pairs:", sorted(stats.items(), key=lambda x: -x[1])[:5])

In [ ]:
# @title Task 2 — BPE: Merge Step
# Given the most frequent pair, merge it everywhere in the vocabulary.

import re

def merge_vocab(pair: tuple, vocab: dict) -> dict:
    """
    Replace all occurrences of `pair` in vocab keys with the merged token.
    
    Args:
        pair: tuple of two adjacent symbols to merge, e.g. ('e', 's').
        vocab: current vocabulary dict.
    Returns:
        new_vocab: updated vocabulary with pair merged.
    """
    new_vocab = {}
    
    # --- MISSING CODE ---
    # Build a regex pattern that matches the exact pair (with space between)
    # but NOT when part of a larger token.
    # Use re.escape for safety. Replace matched pattern with joined pair (no space).
    # Hint: bigram = re.escape(' '.join(pair))
    #       pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    
    return new_vocab

# Run 3 BPE merge iterations
num_merges = 3
for i in range(num_merges):
    stats = get_stats(vocab)
    if not stats:
        break
    best_pair = max(stats, key=stats.get)
    vocab = merge_vocab(best_pair, vocab)
    print(f"Merge {i+1}: {best_pair} → {''.join(best_pair)}")
print("\nVocab after merges:", vocab)

In [ ]:
# @title Task 3 — Manual Softmax
# Softmax converts raw logits into a probability distribution.
# Formula: softmax(x_i) = exp(x_i - max(x)) / sum(exp(x_j - max(x)))
# Subtracting max(x) is the numerical stability trick.

import numpy as np

def softmax(x: np.ndarray) -> np.ndarray:
    """
    Numerically stable softmax over the last axis.
    
    Args:
        x: numpy array of shape (..., vocab_size)
    Returns:
        probabilities: same shape as x, values in (0,1), summing to 1 along last axis.
    """
    # --- MISSING CODE ---
    # 1. Subtract max along last axis (keepdims=True) for numerical stability.
    # 2. Compute exp of shifted values.
    # 3. Divide by sum along last axis (keepdims=True).
    # 4. Return result.
    pass

# Verification
logits = np.array([2.0, 1.0, 0.1, -1.0])
probs = softmax(logits)
print("Logits:", logits)
print("Probabilities:", probs)
print(f"Sum: {probs.sum():.6f}  (should be 1.0)")
print(f"All positive: {np.all(probs > 0)}")

In [ ]:
# @title Task 4 — Scaled Dot-Product Attention
# The core of every Transformer layer.
# Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V

import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    mask: torch.Tensor = None
):
    """
    Args:
        Q: (batch, seq_q, d_k)
        K: (batch, seq_k, d_k)
        V: (batch, seq_k, d_v)
        mask: optional boolean tensor, True = positions to MASK (set to -inf)
    Returns:
        output: (batch, seq_q, d_v)
        attn_weights: (batch, seq_q, seq_k)  ← used in grader
    """
    d_k = Q.size(-1)
    
    # --- MISSING CODE ---
    # 1. Compute raw scores: Q @ K.transpose(-2, -1), scaled by sqrt(d_k).
    # 2. If mask is not None, set masked positions to float('-inf').
    # 3. Apply softmax over last dimension to get attn_weights.
    # 4. Compute output: attn_weights @ V.
    # 5. Return (output, attn_weights).
    pass

# Test
batch, seq, d_k, d_v = 2, 5, 64, 64
Q = torch.randn(batch, seq, d_k)
K = torch.randn(batch, seq, d_k)
V = torch.randn(batch, seq, d_v)

output, attn_weights = scaled_dot_product_attention(Q, K, V)
print(f"Output shape:       {output.shape}")
print(f"Attn weights shape: {attn_weights.shape}")
print(f"Weights sum (row 0): {attn_weights[0, 0].sum().item():.4f}  (should ≈ 1.0)")

In [ ]:
# @title Task 5 — Mixture of Experts: Top-1 Routing
# MoE layers route each token to the single best expert.
# Router: linear projection → softmax → argmax selects the expert.

import torch
import torch.nn as nn

class MoETop1Router(nn.Module):
    def __init__(self, d_model: int, num_experts: int):
        super().__init__()
        self.num_experts = num_experts
        # Learnable gate: maps token embedding → expert logits
        self.gate = nn.Linear(d_model, num_experts, bias=False)

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            expert_indices: (batch, seq_len)  — index of selected expert per token
            router_probs:   (batch, seq_len, num_experts)  — full softmax distribution
        """
        # --- MISSING CODE ---
        # 1. Pass x through self.gate to get logits of shape (batch, seq_len, num_experts).
        # 2. Apply softmax over the last dimension → router_probs.
        # 3. Take argmax over last dimension → expert_indices.
        # 4. Return (expert_indices, router_probs).
        pass

# Test
d_model, num_experts = 128, 8
router = MoETop1Router(d_model, num_experts)
x = torch.randn(2, 10, d_model)  # batch=2, seq=10

expert_indices, router_probs = router(x)
print(f"Expert indices shape: {expert_indices.shape}")
print(f"Router probs shape:   {router_probs.shape}")
print(f"Expert indices (batch 0): {expert_indices[0].tolist()}")
print(f"All indices valid (0–{num_experts-1}): {expert_indices.max().item() < num_experts}")

In [ ]:
# @title 🔍 Run to Verify Section 1 { display-mode: "form" }
import numpy as np, torch

print("--- Section 1 Grader ---")

# Task 1: BPE stats returns a dict
assert isinstance(stats, dict), "Task 1: get_stats() must return a dict"
assert len(stats) > 0, "Task 1: stats dict is empty — check get_stats()"
print("✅ Task 1 passed: BPE stats dict populated")

# Task 2: merge_vocab reduces unique symbols
test_vocab = get_vocab(["low", "low", "lower"])
test_stats = get_stats(test_vocab)
best = max(test_stats, key=test_stats.get)
merged = merge_vocab(best, test_vocab)
assert isinstance(merged, dict), "Task 2: merge_vocab() must return a dict"
assert merged != test_vocab, "Task 2: merge produced no change — check regex logic"
print("✅ Task 2 passed: merge_vocab produces updated vocabulary")

# Task 3: softmax
x = np.array([1.0, 2.0, 3.0])
out = softmax(x)
assert out is not None, "Task 3: softmax returned None"
assert abs(out.sum() - 1.0) < 1e-5, "Task 3: softmax does not sum to 1"
assert np.all(out > 0), "Task 3: softmax contains non-positive values"
print("✅ Task 3 passed: softmax correct")

# Task 4: attention shapes
Q2 = torch.randn(1, 4, 32); K2 = torch.randn(1, 4, 32); V2 = torch.randn(1, 4, 32)
o2, aw2 = scaled_dot_product_attention(Q2, K2, V2)
assert o2 is not None, "Task 4: attention output is None"
assert tuple(aw2.shape) == (1, 4, 4), f"Task 4: attn_weights shape {aw2.shape} != (1,4,4)"
assert abs(aw2[0, 0].sum().item() - 1.0) < 1e-4, "Task 4: attention weights don't sum to 1"
print("✅ Task 4 passed: attention shapes and sums correct")

# Task 5: MoE routing
test_router = MoETop1Router(64, 4)
tx = torch.randn(1, 6, 64)
ei, rp = test_router(tx)
assert ei is not None, "Task 5: expert_indices is None"
assert tuple(ei.shape) == (1, 6), f"Task 5: expert_indices shape {ei.shape} != (1,6)"
assert tuple(rp.shape) == (1, 6, 4), f"Task 5: router_probs shape {rp.shape} != (1,6,4)"
assert ei.max().item() < 4, "Task 5: expert index out of bounds"
print("✅ Task 5 passed: MoE routing shapes and bounds correct")

print("\n🎉 Section 1 complete!")

---
## Section 2: Quantization
**30 minutes | Tasks 6–10**

Reducing model memory footprint via quantization strategies supported by BitsAndBytes.

In [ ]:
# @title Task 6 — FP16 Memory Math
# A model's parameter memory = num_params * bytes_per_param.
# FP32 = 4 bytes/param | FP16/BF16 = 2 bytes/param | INT8 = 1 | INT4 = 0.5

def compute_model_memory_gb(
    num_params_millions: float,
    dtype_bytes: float
) -> float:
    """
    Compute memory in GB for a model of given size and dtype.
    
    Args:
        num_params_millions: parameter count in millions (e.g., 7000 for 7B).
        dtype_bytes: bytes per parameter (4=fp32, 2=fp16/bf16, 1=int8, 0.5=int4).
    Returns:
        memory_gb: float
    """
    # --- MISSING CODE ---
    # 1. Convert millions to actual count: num_params = num_params_millions * 1e6
    # 2. Memory in bytes = num_params * dtype_bytes
    # 3. Convert to GB: / 1e9
    # 4. Return memory_gb
    pass

# Compute and store results for verification
opt_125m_fp32_gb = compute_model_memory_gb(125, 4)
opt_125m_fp16_gb = compute_model_memory_gb(125, 2)
opt_125m_int4_gb = compute_model_memory_gb(125, 0.5)

llama7b_fp16_gb  = compute_model_memory_gb(7000, 2)
llama7b_int4_gb  = compute_model_memory_gb(7000, 0.5)

print(f"OPT-125M  | FP32: {opt_125m_fp32_gb:.2f} GB | FP16: {opt_125m_fp16_gb:.2f} GB | INT4: {opt_125m_int4_gb:.2f} GB")
print(f"LLaMA-7B  | FP16: {llama7b_fp16_gb:.2f} GB | INT4: {llama7b_int4_gb:.2f} GB")
print(f"\nT4 VRAM budget (16GB):")
print(f"  LLaMA-7B FP16 fits? {llama7b_fp16_gb <= 16}")
print(f"  LLaMA-7B INT4 fits? {llama7b_int4_gb <= 16}")

In [ ]:
# @title Task 7 — 4-bit NF4 BitsAndBytes Configuration
# NF4 (NormalFloat4) is the default 4-bit dtype in QLoRA/BitsAndBytes.
# It quantizes weights to 4-bit, dramatically reducing memory.

from transformers import BitsAndBytesConfig
import torch

# --- MISSING CODE ---
# Create a BitsAndBytesConfig object named `bnb_config_nf4` with:
#   load_in_4bit = True
#   bnb_4bit_quant_type = "nf4"
#   bnb_4bit_compute_dtype = torch.bfloat16   (Task 8 will explain why bf16)
#   bnb_4bit_use_double_quant = False          (Task 10 enables this)
# bnb_config_nf4 = BitsAndBytesConfig(...)

print("BitsAndBytes NF4 config:")
print(f"  load_in_4bit:           {bnb_config_nf4.load_in_4bit}")
print(f"  bnb_4bit_quant_type:    {bnb_config_nf4.bnb_4bit_quant_type}")
print(f"  bnb_4bit_compute_dtype: {bnb_config_nf4.bnb_4bit_compute_dtype}")
print(f"  double_quant:           {bnb_config_nf4.bnb_4bit_use_double_quant}")

In [ ]:
# @title Task 8 — Compute Dtype: BF16 vs FP16
# Even when weights are stored in 4-bit, actual matrix multiplications
# happen in a higher-precision compute dtype.
# BF16: same exponent range as FP32, less precision. Better for training stability.
# FP16: higher precision, narrower range — can overflow on T4.

import torch

def select_compute_dtype(gpu_name: str) -> torch.dtype:
    """
    Choose the optimal compute dtype based on GPU capabilities.
    
    Rules:
      - Ampere+ (A100, A10G, 30xx, 40xx): torch.bfloat16
      - Older cards (T4, V100, 16xx):     torch.float16
    
    Args:
        gpu_name: string from torch.cuda.get_device_name(0)
    Returns:
        torch.dtype
    """
    # --- MISSING CODE ---
    # Check if any Ampere-class identifier is in gpu_name (case-insensitive).
    # Ampere identifiers: 'A100', 'A10G', 'A10', 'RTX 30', 'RTX 40', '3090', '4090'
    # Return torch.bfloat16 if Ampere+, else torch.float16.
    pass

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Tesla T4"
compute_dtype = select_compute_dtype(gpu)
print(f"GPU detected:    {gpu}")
print(f"Compute dtype:   {compute_dtype}")
print(f"Expected for T4: torch.float16")

In [ ]:
# @title Task 9 — VRAM Profiling with torch.cuda
# Load OPT-125M in 4-bit and measure actual VRAM usage.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "facebook/opt-125m"

def profile_vram_usage(model_id: str, quantization_config=None) -> dict:
    """
    Load a model, measure VRAM before and after, return delta.
    
    Args:
        model_id: HuggingFace model identifier.
        quantization_config: optional BitsAndBytesConfig.
    Returns:
        dict with keys: 'vram_before_gb', 'vram_after_gb', 'delta_gb', 'model'
    """
    # --- MISSING CODE ---
    # 1. torch.cuda.empty_cache() to clear stale allocations.
    # 2. Record vram_before = torch.cuda.memory_reserved(0) / 1e9
    # 3. Load model: AutoModelForCausalLM.from_pretrained(
    #       model_id, quantization_config=quantization_config, device_map="auto")
    # 4. Record vram_after = torch.cuda.memory_reserved(0) / 1e9
    # 5. Return dict with vram_before_gb, vram_after_gb, delta_gb, model.
    pass

result = profile_vram_usage(MODEL_ID, quantization_config=bnb_config_nf4)
model_4bit = result['model']

print(f"VRAM before load: {result['vram_before_gb']:.3f} GB")
print(f"VRAM after load:  {result['vram_after_gb']:.3f} GB")
print(f"Model delta:      {result['delta_gb']:.3f} GB")
print(f"\nTheoretical INT4 size: {compute_model_memory_gb(125, 0.5):.3f} GB")

In [ ]:
# @title Task 10 — Double Quantization Setup
# Double quantization quantizes the quantization constants themselves,
# saving ~0.37 bits per parameter additionally (QLoRA paper).

from transformers import BitsAndBytesConfig
import torch

# --- MISSING CODE ---
# Create bnb_config_dq (double quantization config):
#   Same as bnb_config_nf4, but with bnb_4bit_use_double_quant = True
# bnb_config_dq = BitsAndBytesConfig(...)

# Compute theoretical savings
base_4bit_gb = compute_model_memory_gb(7000, 0.5)
# Double quant saves ~0.37 bits/param on quantization constants
# Approximation: savings ≈ num_params * 0.37 / 8 bytes → GB
dq_savings_gb = 7000e6 * 0.37 / 8 / 1e9

print(f"NF4 config double_quant:  {bnb_config_nf4.bnb_4bit_use_double_quant}")
print(f"DQ  config double_quant:  {bnb_config_dq.bnb_4bit_use_double_quant}")
print(f"\nFor 7B model:")
print(f"  Base 4-bit:         {base_4bit_gb:.2f} GB")
print(f"  Double quant saves: {dq_savings_gb:.2f} GB")
print(f"  Effective size:     {base_4bit_gb - dq_savings_gb:.2f} GB")

In [ ]:
# @title 🔍 Run to Verify Section 2 { display-mode: "form" }
import torch
from transformers import BitsAndBytesConfig

print("--- Section 2 Grader ---")

# Task 6
assert opt_125m_fp16_gb is not None, "Task 6: compute_model_memory_gb returned None"
assert abs(opt_125m_fp16_gb - 0.25) < 0.01, f"Task 6: OPT-125M FP16 should be ~0.25 GB, got {opt_125m_fp16_gb}"
assert abs(llama7b_fp16_gb - 14.0) < 0.5, f"Task 6: LLaMA-7B FP16 should be ~14 GB, got {llama7b_fp16_gb}"
print("✅ Task 6 passed: memory calculations correct")

# Task 7
assert bnb_config_nf4.load_in_4bit == True, "Task 7: load_in_4bit must be True"
assert bnb_config_nf4.bnb_4bit_quant_type == "nf4", "Task 7: quant_type must be 'nf4'"
assert bnb_config_nf4.bnb_4bit_compute_dtype == torch.bfloat16, "Task 7: compute_dtype must be bfloat16"
print("✅ Task 7 passed: BitsAndBytes NF4 config correct")

# Task 8
assert compute_dtype is not None, "Task 8: select_compute_dtype returned None"
assert compute_dtype in [torch.float16, torch.bfloat16], "Task 8: must return a torch float dtype"
t4_dtype = select_compute_dtype("Tesla T4")
assert t4_dtype == torch.float16, f"Task 8: T4 should return float16, got {t4_dtype}"
a100_dtype = select_compute_dtype("A100")
assert a100_dtype == torch.bfloat16, f"Task 8: A100 should return bfloat16, got {a100_dtype}"
print("✅ Task 8 passed: compute dtype selection correct")

# Task 9
assert model_4bit is not None, "Task 9: model_4bit not loaded"
assert result['delta_gb'] is not None, "Task 9: delta_gb not computed"
assert result['delta_gb'] > 0, "Task 9: VRAM delta should be positive"
print(f"✅ Task 9 passed: model loaded, VRAM delta = {result['delta_gb']:.3f} GB")

# Task 10
assert bnb_config_dq.bnb_4bit_use_double_quant == True, "Task 10: double_quant must be True"
assert bnb_config_dq.bnb_4bit_quant_type == "nf4", "Task 10: quant_type must still be nf4"
print("✅ Task 10 passed: double quantization config correct")

print("\n🎉 Section 2 complete!")

---
## Section 3: Inference & Prompting
**30 minutes | Tasks 11–15**

Control decoding behavior and structure prompts for precise outputs.

In [ ]:
# @title Task 11 — GGUF Model Loading (ctransformers)
# GGUF is a binary format for quantized LLMs, used by llama.cpp and ctransformers.
# TinyLlama is a compact 1.1B model — fits in < 1GB in Q4 quantization.

from huggingface_hub import hf_hub_download
from ctransformers import AutoModelForCausalLM as CT_AutoModel

GGUF_REPO = "TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF"
GGUF_FILE = "tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf"

# Download the GGUF file from HuggingFace Hub
gguf_path = hf_hub_download(repo_id=GGUF_REPO, filename=GGUF_FILE)
print(f"GGUF file path: {gguf_path}")

# --- MISSING CODE ---
# Load the GGUF model using ctransformers:
#   gguf_model = CT_AutoModel.from_pretrained(
#       gguf_path,
#       model_type="llama",
#       gpu_layers=50          # offload layers to GPU
#   )
# Store the result in `gguf_model`.

print(f"\nModel type: {type(gguf_model)}")
test_out = gguf_model("Hello", max_new_tokens=5)
print(f"Quick test output: '{test_out[:50]}'")

In [ ]:
# @title Task 12 — Temperature Scaling
# Temperature T scales logits before softmax: adjusted_logits = logits / T
# T < 1 → sharper distribution (more deterministic)
# T > 1 → flatter distribution (more random)
# T = 1 → unmodified distribution

import numpy as np

def temperature_scaled_probs(logits: np.ndarray, temperature: float) -> np.ndarray:
    """
    Apply temperature scaling and return softmax probabilities.
    
    Args:
        logits: raw model logits, shape (vocab_size,)
        temperature: float > 0
    Returns:
        probs: probability distribution, shape (vocab_size,)
    """
    # --- MISSING CODE ---
    # 1. Guard: if temperature <= 0, raise ValueError.
    # 2. Divide logits by temperature.
    # 3. Apply softmax (reuse your Task 3 implementation or np directly).
    # 4. Return probs.
    pass

logits = np.array([3.0, 1.5, 0.5, -1.0, -2.0])
tokens = ["cat", "dog", "bird", "fish", "rock"]

for T in [0.3, 1.0, 2.0]:
    p = temperature_scaled_probs(logits, T)
    print(f"T={T}: {dict(zip(tokens, [f'{v:.3f}' for v in p]))}")

print("\nT=0.3 top token more dominant:", temperature_scaled_probs(logits, 0.3).argmax() == 0)

In [ ]:
# @title Task 13 — Top-P (Nucleus) Sampling
# Top-P sampling: keep the smallest set of tokens whose cumulative probability ≥ p.
# Zeroes out tokens outside the nucleus, then renormalizes.

import numpy as np

def top_p_filter(probs: np.ndarray, p: float) -> np.ndarray:
    """
    Zero out tokens outside the top-p nucleus and renormalize.
    
    Args:
        probs: probability array (already softmaxed), shape (vocab_size,)
        p: nucleus threshold, float in (0, 1]
    Returns:
        filtered_probs: renormalized array with same shape
    """
    # --- MISSING CODE ---
    # 1. Sort indices by descending probability.
    # 2. Compute cumulative sum of sorted probabilities.
    # 3. Find the cutoff: smallest set where cumsum >= p.
    #    Hint: np.cumsum, then find the index where cumsum first >= p.
    # 4. Zero out all tokens NOT in this nucleus.
    # 5. Renormalize: divide by sum.
    # 6. Return filtered_probs.
    pass

probs = softmax(np.array([5.0, 3.0, 2.0, 1.0, 0.5, 0.1, 0.05, 0.01]))
tokens = ["A", "B", "C", "D", "E", "F", "G", "H"]

for p_val in [0.5, 0.9, 1.0]:
    filtered = top_p_filter(probs, p_val)
    active = sum(1 for v in filtered if v > 0)
    print(f"p={p_val}: {active} active tokens, sum={filtered.sum():.4f}")

In [ ]:
# @title Task 14 — Few-Shot Prompt Engineering
# Few-shot prompting provides worked examples inside the prompt.
# The model learns the desired input→output pattern from these examples.

def build_few_shot_prompt(
    task_instruction: str,
    examples: list[dict],  # each: {"input": str, "output": str}
    new_input: str
) -> str:
    """
    Assemble a few-shot prompt string.
    
    Format:
        <task_instruction>
        
        Input: <example_input>
        Output: <example_output>
        ... (repeated for each example)
        
        Input: <new_input>
        Output:
    
    Returns:
        prompt: str
    """
    # --- MISSING CODE ---
    # 1. Start with task_instruction + two newlines.
    # 2. For each example dict, append "Input: {input}\nOutput: {output}\n\n".
    # 3. Append "Input: {new_input}\nOutput:".
    # 4. Return the full prompt string.
    pass

few_shot_prompt = build_few_shot_prompt(
    task_instruction="Classify the sentiment of each review as Positive or Negative.",
    examples=[
        {"input": "The product was fantastic and exceeded expectations.", "output": "Positive"},
        {"input": "Terrible quality, broke after one day.", "output": "Negative"},
        {"input": "Exactly what I needed, fast shipping too.", "output": "Positive"},
    ],
    new_input="It stopped working within a week and support was unhelpful."
)

print(few_shot_prompt)
print(f"\nPrompt contains 'Output:' x{few_shot_prompt.count('Output:')} times (should be 4)")

In [ ]:
# @title Task 15 — Chain-of-Thought (CoT) Reasoning Template
# CoT prompting instructs the model to reason step-by-step before answering.
# Classic trigger phrase: "Let's think step by step."

def build_cot_prompt(
    question: str,
    system_role: str = "You are a helpful assistant that reasons step by step."
) -> str:
    """
    Build a Chain-of-Thought prompt in ChatML format.
    
    Format:
        <|system|>
        {system_role}
        <|user|>
        {question}
        Let's think step by step.
        <|assistant|>
    
    Args:
        question: the reasoning question.
        system_role: system instruction string.
    Returns:
        cot_prompt: str
    """
    # --- MISSING CODE ---
    # Assemble the four-part ChatML structure shown above.
    # Append "Let's think step by step." after the question.
    # End with <|assistant|> on its own line (model will continue from here).
    pass

cot_prompt = build_cot_prompt(
    question="If a train travels 120 km in 1.5 hours, then takes a 20-minute break, "
             "and covers another 90 km in 1 hour, what is its average speed for the entire journey?"
)
print(cot_prompt)

# Run inference with gguf_model
cot_response = gguf_model(cot_prompt, max_new_tokens=200, temperature=0.7)
print("\n--- Model Response ---")
print(cot_response)

In [ ]:
# @title 🔍 Run to Verify Section 3 { display-mode: "form" }
import numpy as np

print("--- Section 3 Grader ---")

# Task 11
assert gguf_model is not None, "Task 11: gguf_model not loaded"
print("✅ Task 11 passed: GGUF model loaded")

# Task 12
t_logits = np.array([2.0, 1.0, 0.0])
p_t1 = temperature_scaled_probs(t_logits, 1.0)
p_t01 = temperature_scaled_probs(t_logits, 0.1)
assert p_t1 is not None, "Task 12: returned None"
assert abs(p_t1.sum() - 1.0) < 1e-5, "Task 12: probs don't sum to 1"
assert p_t01[0] > p_t1[0], "Task 12: low temp should sharpen top token probability"
try:
    temperature_scaled_probs(t_logits, 0)
    assert False, "Task 12: should raise ValueError for T=0"
except ValueError:
    pass
print("✅ Task 12 passed: temperature scaling correct")

# Task 13
test_probs = softmax(np.array([5.0, 1.0, 0.1, 0.01]))
f_p05 = top_p_filter(test_probs, 0.5)
f_p10 = top_p_filter(test_probs, 1.0)
assert f_p05 is not None, "Task 13: returned None"
assert abs(f_p05.sum() - 1.0) < 1e-5, "Task 13: filtered probs don't sum to 1"
assert sum(1 for v in f_p05 if v > 0) <= sum(1 for v in f_p10 if v > 0), \
    "Task 13: smaller p should produce smaller or equal nucleus"
print("✅ Task 13 passed: top-p filtering correct")

# Task 14
assert few_shot_prompt is not None, "Task 14: build_few_shot_prompt returned None"
assert few_shot_prompt.count("Output:") == 4, \
    f"Task 14: expected 4 'Output:' occurrences (3 examples + 1 new), got {few_shot_prompt.count('Output:')}"
assert few_shot_prompt.strip().endswith("Output:"), \
    "Task 14: prompt must end with 'Output:' for the new input"
print("✅ Task 14 passed: few-shot prompt structure correct")

# Task 15
assert cot_prompt is not None, "Task 15: build_cot_prompt returned None"
assert "<|system|>" in cot_prompt, "Task 15: missing <|system|> tag"
assert "<|user|>" in cot_prompt, "Task 15: missing <|user|> tag"
assert "<|assistant|>" in cot_prompt, "Task 15: missing <|assistant|> tag"
assert "step by step" in cot_prompt.lower(), "Task 15: missing CoT trigger phrase"
print("✅ Task 15 passed: CoT prompt structure correct")

print("\n🎉 Section 3 complete!")

---
## Section 4: RAG with FAISS
**45 minutes | Tasks 16–20**

Retrieval-Augmented Generation: load documents, chunk, embed, index, and query.

In [ ]:
# @title Task 16 — Web Document Loading (LangChain)
# LangChain's WebBaseLoader fetches and parses HTML pages into Document objects.

from langchain_community.document_loaders import WebBaseLoader

URLS = [
    "https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)",
    "https://en.wikipedia.org/wiki/Large_language_model",
]

def load_web_documents(urls: list[str]) -> list:
    """
    Load web pages and return a list of LangChain Document objects.
    
    Args:
        urls: list of URL strings.
    Returns:
        docs: list of Document objects (each has .page_content and .metadata).
    """
    # --- MISSING CODE ---
    # 1. Instantiate WebBaseLoader with the urls list.
    # 2. Call loader.load() to fetch and parse the pages.
    # 3. Return the docs list.
    pass

raw_docs = load_web_documents(URLS)
print(f"Loaded {len(raw_docs)} documents")
for i, doc in enumerate(raw_docs):
    print(f"  Doc {i+1}: {len(doc.page_content)} chars | Source: {doc.metadata.get('source', 'N/A')[:60]}")

In [ ]:
# @title Task 17 — Recursive Character Text Splitting
# Large documents must be chunked before embedding.
# RecursiveCharacterTextSplitter tries to split on paragraph boundaries first,
# then sentences, then words, before falling back to characters.

from langchain.text_splitter import RecursiveCharacterTextSplitter

def split_documents(docs: list, chunk_size: int = 512, chunk_overlap: int = 64) -> list:
    """
    Split documents into overlapping chunks.
    
    Args:
        docs: list of LangChain Document objects.
        chunk_size: maximum characters per chunk.
        chunk_overlap: number of overlapping characters between adjacent chunks.
    Returns:
        chunks: list of Document objects (more numerous, shorter).
    """
    # --- MISSING CODE ---
    # 1. Instantiate RecursiveCharacterTextSplitter(
    #       chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    # 2. Call splitter.split_documents(docs).
    # 3. Return chunks.
    pass

chunks = split_documents(raw_docs, chunk_size=512, chunk_overlap=64)
print(f"Original docs: {len(raw_docs)}")
print(f"Chunks produced: {len(chunks)}")
print(f"\nSample chunk (first 200 chars):\n{chunks[0].page_content[:200]}")

In [ ]:
# @title Task 18 — HuggingFace Sentence Embeddings
# Embeddings map text chunks to dense vectors.
# sentence-transformers/all-MiniLM-L6-v2 produces 384-dim vectors, runs fast on T4.

from langchain_community.embeddings import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

def create_embeddings(model_name: str) -> HuggingFaceEmbeddings:
    """
    Instantiate a HuggingFaceEmbeddings object.
    
    Args:
        model_name: HuggingFace model identifier for the sentence transformer.
    Returns:
        embeddings: HuggingFaceEmbeddings instance.
    """
    # --- MISSING CODE ---
    # Instantiate HuggingFaceEmbeddings with:
    #   model_name=model_name
    #   model_kwargs={'device': 'cuda'}
    #   encode_kwargs={'normalize_embeddings': True}
    pass

embeddings = create_embeddings(EMBEDDING_MODEL)

# Test embedding a single sentence
test_vec = embeddings.embed_query("What is attention mechanism in transformers?")
print(f"Embedding model:       {EMBEDDING_MODEL}")
print(f"Embedding dimension:   {len(test_vec)}")
print(f"First 5 values:        {[f'{v:.4f}' for v in test_vec[:5]]}")

In [ ]:
# @title Task 19 — FAISS Vector Index
# FAISS (Facebook AI Similarity Search) enables fast approximate nearest-neighbor
# lookup over millions of vectors using L2 or cosine distance.

from langchain_community.vectorstores import FAISS

def build_faiss_index(chunks: list, embeddings) -> FAISS:
    """
    Embed all chunks and build a FAISS index.
    
    Args:
        chunks: list of Document objects.
        embeddings: HuggingFaceEmbeddings instance.
    Returns:
        vector_db: FAISS vectorstore.
    """
    # --- MISSING CODE ---
    # Use FAISS.from_documents(chunks, embeddings) to build the index.
    # Return the resulting FAISS object as `vector_db`.
    pass

print("Building FAISS index... (this may take ~1-2 minutes)")
vector_db = build_faiss_index(chunks, embeddings)

# Quick similarity search test
test_results = vector_db.similarity_search("How does the attention mechanism work?", k=2)
print(f"\nFAISS index built: {vector_db.index.ntotal} vectors indexed")
print(f"\nTop result snippet (first 150 chars):")
print(test_results[0].page_content[:150])

In [ ]:
# @title Task 20 — RetrievalQA Chain Assembly
# Assemble a full RAG pipeline: retriever → LLM → answer.
# We'll use the HuggingFace pipeline wrapper for the quantized OPT-125M model.

from langchain.chains import RetrievalQA
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, pipeline

# Build the LLM wrapper around model_4bit
tokenizer_opt = AutoTokenizer.from_pretrained("facebook/opt-125m")
hf_pipeline = pipeline(
    "text-generation",
    model=model_4bit,
    tokenizer=tokenizer_opt,
    max_new_tokens=128,
    do_sample=False
)
llm_langchain = HuggingFacePipeline(pipeline=hf_pipeline)

def build_retrieval_qa(
    llm,
    vector_db: FAISS,
    k: int = 3
) -> RetrievalQA:
    """
    Assemble a RetrievalQA chain.
    
    Args:
        llm: LangChain LLM instance.
        vector_db: FAISS vectorstore.
        k: number of documents to retrieve per query.
    Returns:
        qa_chain: RetrievalQA instance.
    """
    # --- MISSING CODE ---
    # 1. Create retriever = vector_db.as_retriever(search_kwargs={"k": k})
    # 2. Create qa_chain = RetrievalQA.from_chain_type(
    #       llm=llm,
    #       chain_type="stuff",
    #       retriever=retriever,
    #       return_source_documents=True
    #    )
    # 3. Return qa_chain.
    pass

qa_chain = build_retrieval_qa(llm_langchain, vector_db, k=3)

result = qa_chain.invoke({"query": "What is the Transformer architecture?"})
print("Query: What is the Transformer architecture?")
print("\nAnswer:", result['result'][:300])
print(f"\nSources retrieved: {len(result['source_documents'])}")

In [ ]:
# @title 🔍 Run to Verify Section 4 { display-mode: "form" }
from langchain_community.vectorstores import FAISS as _FAISS
from langchain.chains import RetrievalQA as _RetrievalQA

print("--- Section 4 Grader ---")

# Task 16
assert raw_docs is not None, "Task 16: raw_docs is None"
assert len(raw_docs) >= 2, f"Task 16: expected ≥2 docs, got {len(raw_docs)}"
assert hasattr(raw_docs[0], 'page_content'), "Task 16: documents missing page_content attribute"
assert len(raw_docs[0].page_content) > 100, "Task 16: document content suspiciously short"
print(f"✅ Task 16 passed: {len(raw_docs)} documents loaded")

# Task 17
assert chunks is not None, "Task 17: chunks is None"
assert len(chunks) > len(raw_docs), "Task 17: chunks should be more numerous than raw_docs"
assert all(len(c.page_content) <= 600 for c in chunks), \
    "Task 17: some chunks exceed chunk_size significantly"
print(f"✅ Task 17 passed: {len(chunks)} chunks produced")

# Task 18
assert embeddings is not None, "Task 18: embeddings object is None"
test_v = embeddings.embed_query("test")
assert len(test_v) == 384, f"Task 18: expected 384-dim embeddings, got {len(test_v)}"
print(f"✅ Task 18 passed: {len(test_v)}-dim embeddings verified")

# Task 19
assert vector_db is not None, "Task 19: vector_db is None"
assert isinstance(vector_db, _FAISS), f"Task 19: expected FAISS instance, got {type(vector_db)}"
assert vector_db.index.ntotal == len(chunks), \
    f"Task 19: FAISS index has {vector_db.index.ntotal} vectors, expected {len(chunks)}"
print(f"✅ Task 19 passed: FAISS index with {vector_db.index.ntotal} vectors")

# Task 20
assert qa_chain is not None, "Task 20: qa_chain is None"
assert isinstance(qa_chain, _RetrievalQA), f"Task 20: expected RetrievalQA, got {type(qa_chain)}"
assert result is not None and 'result' in result, "Task 20: chain invoke did not return 'result' key"
assert 'source_documents' in result, "Task 20: return_source_documents=True not working"
print(f"✅ Task 20 passed: RetrievalQA chain returns answer + {len(result['source_documents'])} sources")

print("\n🎉 Section 4 complete!")

---
## Section 5: Agents
**30 minutes | Tasks 21–25**

Build a ReAct agent with custom tools, multi-step reasoning, and a tuned persona.

In [ ]:
# @title Task 21 — Python Function Tool Definition
# LangChain tools are Python functions decorated with @tool.
# The docstring IS the tool description — the agent reads it to decide when to use the tool.

from langchain.tools import tool

@tool
def get_current_date(query: str) -> str:
    """Returns today's date in YYYY-MM-DD format. Use when asked about current date or time."""
    from datetime import date
    return str(date.today())

# --- MISSING CODE ---
# Define a second @tool function named `word_counter` that:
#   - Takes a single string argument `text: str`
#   - Returns a string: "Word count: {N}" where N is the number of whitespace-separated words.
#   - Has a clear docstring (the agent will read this to know when to invoke it).
#
# Example: word_counter("hello world foo") → "Word count: 3"

print("Tools defined:")
print(f"  {get_current_date.name}: {get_current_date.description}")
print(f"  {word_counter.name}: {word_counter.description}")

# Sanity test
print(f"\nDate tool test:  {get_current_date.run('today')}")
print(f"Counter test:    {word_counter.run('The quick brown fox')}")

In [ ]:
# @title Task 22 — Math Tool Integration
# Build a more capable math tool that handles basic arithmetic safely.

from langchain.tools import tool
import ast
import operator

@tool
def safe_calculator(expression: str) -> str:
    """
    Evaluates a safe arithmetic expression string and returns the result.
    Supports: +, -, *, /, **, //, %. Input must be a valid Python arithmetic expression.
    Examples: '2 + 3 * 4', '(100 / 4) ** 2', '17 % 5'
    """
    # --- MISSING CODE ---
    # Implement a safe evaluator:
    # 1. Define SAFE_OPERATORS: mapping of ast node types to operator functions.
    #    Include: ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow, ast.FloorDiv, ast.Mod
    #    Map to: operator.add, operator.sub, operator.mul, operator.truediv,
    #            operator.pow, operator.floordiv, operator.mod
    # 2. Define a recursive eval_node(node) function:
    #    - ast.Constant (or ast.Num for older Python): return node.n or node.value
    #    - ast.BinOp: look up op type in SAFE_OPERATORS, recursively eval left+right
    #    - ast.UnaryOp with ast.USub: return -eval_node(node.operand)
    #    - else: raise ValueError("Unsafe operation")
    # 3. Parse expression with ast.parse(expression, mode='eval')
    # 4. Return str(eval_node(tree.body))
    # 5. Wrap in try/except; return "Error: {msg}" on failure.
    pass

print("Math tool tests:")
print(f"  '2 + 3 * 4'     → {safe_calculator.run('2 + 3 * 4')}")
print(f"  '(10 / 4) ** 2' → {safe_calculator.run('(10 / 4) ** 2')}")
print(f"  '17 % 5'        → {safe_calculator.run('17 % 5')}")
print(f"  '__import__...' → {safe_calculator.run('__import__(os)')}")

In [ ]:
# @title Task 23 — Zero-Shot ReAct Agent Initialization
# ReAct (Reason + Act): agent interleaves Thought → Action → Observation loops.
# Zero-shot: no examples given; the agent decides tool use from descriptions alone.

from langchain.agents import initialize_agent, AgentType
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, pipeline

# Re-use gguf_model with a LangChain-compatible wrapper
# We'll use a custom LLM wrapper since ctransformers isn't natively in LC
from langchain.llms.base import LLM
from typing import Optional, List

class GGUFLangChainLLM(LLM):
    model: object = None
    max_new_tokens: int = 256
    temperature: float = 0.1

    class Config:
        arbitrary_types_allowed = True

    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        out = self.model(prompt, max_new_tokens=self.max_new_tokens, temperature=self.temperature)
        if stop:
            for s in stop:
                if s in out:
                    out = out[:out.index(s)]
        return out

    @property
    def _llm_type(self) -> str:
        return "gguf_tinyllama"

agent_llm = GGUFLangChainLLM(model=gguf_model, max_new_tokens=256, temperature=0.1)

tools = [get_current_date, word_counter, safe_calculator]

def create_react_agent(llm, tools: list):
    """
    Initialize a Zero-Shot ReAct agent.
    
    Args:
        llm: LangChain LLM instance.
        tools: list of @tool decorated functions.
    Returns:
        agent: initialized AgentExecutor.
    """
    # --- MISSING CODE ---
    # Use initialize_agent with:
    #   tools=tools
    #   llm=llm
    #   agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION
    #   verbose=True
    #   handle_parsing_errors=True
    #   max_iterations=5
    pass

react_agent = create_react_agent(agent_llm, tools)
print(f"Agent type:  {react_agent.agent.__class__.__name__}")
print(f"Tools:       {[t.name for t in react_agent.tools]}")

In [ ]:
# @title Task 24 — Multi-Step Tool Use
# Give the agent a compound task requiring multiple tool calls.
# The agent must plan, select tools, observe results, and synthesize an answer.

def run_multistep_task(agent, query: str) -> dict:
    """
    Run a multi-step reasoning query through the ReAct agent.
    
    Args:
        agent: initialized AgentExecutor.
        query: user question string.
    Returns:
        result: dict with 'output' key containing the final answer.
    """
    # --- MISSING CODE ---
    # Call agent.invoke({"input": query}) and return the result dict.
    # The ReAct loop (Thought/Action/Observation) will print due to verbose=True.
    pass

compound_query = (
    "What is today's date? Also calculate 17 * 24, and tell me how many words "
    "are in the sentence: 'The quick brown fox jumps over the lazy dog'."
)

print(f"Query: {compound_query}\n")
print("=" * 60)
agent_result = run_multistep_task(react_agent, compound_query)
print("=" * 60)
print(f"\nFinal output:\n{agent_result.get('output', 'No output key found')[:500]}")

In [ ]:
# @title Task 25 — Agent System Persona Tuning
# Customize the agent's behavior by injecting a system persona into its prompt.
# LangChain allows prepending a prefix to the ReAct prompt template.

from langchain.agents import initialize_agent, AgentType
from langchain.prompts import PromptTemplate

PERSONA_PREFIX = """
You are ARIA (Automated Reasoning and Inference Assistant), an expert AI agent 
specializing in technical analysis. You are precise, concise, and always show 
your calculations explicitly. When using tools, explain why you chose each tool 
before invoking it. Never guess — always use a tool when factual data is needed.
"""

def create_persona_agent(llm, tools: list, persona_prefix: str):
    """
    Create a ReAct agent with a custom persona injected into the system prompt.
    
    Args:
        llm: LangChain LLM.
        tools: list of tools.
        persona_prefix: string prepended to the agent's prompt.
    Returns:
        persona_agent: AgentExecutor with custom persona.
    """
    # --- MISSING CODE ---
    # Use initialize_agent with:
    #   tools=tools, llm=llm
    #   agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION
    #   verbose=True
    #   handle_parsing_errors=True
    #   max_iterations=5
    #   agent_kwargs={"prefix": persona_prefix}
    pass

aria_agent = create_persona_agent(agent_llm, tools, PERSONA_PREFIX)

# Verify persona is embedded
agent_prompt_text = aria_agent.agent.llm_chain.prompt.template
print("Persona injected into prompt:")
print(agent_prompt_text[:300])
print("...")

# Quick test
aria_result = aria_agent.invoke({"input": "Calculate (144 / 12) + (7 ** 2) and give me the result."})
print(f"\nARIA answer: {aria_result.get('output', '')[:200]}")

In [ ]:
# @title 🔍 Run to Verify Section 5 { display-mode: "form" }
from langchain.agents import AgentExecutor

print("--- Section 5 Grader ---")

# Task 21
assert word_counter is not None, "Task 21: word_counter tool not defined"
assert hasattr(word_counter, 'name'), "Task 21: not a valid @tool object"
wc_result = word_counter.run("one two three four five")
assert "5" in str(wc_result), f"Task 21: word_counter('one two three four five') should contain '5', got: {wc_result}"
print("✅ Task 21 passed: word_counter tool correct")

# Task 22
assert safe_calculator is not None, "Task 22: safe_calculator not defined"
calc_r1 = safe_calculator.run("2 + 3")
assert "5" in str(calc_r1), f"Task 22: 2+3 should be 5, got {calc_r1}"
calc_r2 = safe_calculator.run("10 ** 3")
assert "1000" in str(calc_r2), f"Task 22: 10**3 should be 1000, got {calc_r2}"
unsafe_r = safe_calculator.run("__import__('os').system('ls')")
assert "Error" in str(unsafe_r), f"Task 22: unsafe input should return Error, got {unsafe_r}"
print("✅ Task 22 passed: safe_calculator handles arithmetic and blocks unsafe input")

# Task 23
assert react_agent is not None, "Task 23: react_agent not initialized"
assert isinstance(react_agent, AgentExecutor), f"Task 23: expected AgentExecutor, got {type(react_agent)}"
assert len(react_agent.tools) == 3, f"Task 23: expected 3 tools, got {len(react_agent.tools)}"
tool_names = {t.name for t in react_agent.tools}
assert 'safe_calculator' in tool_names, "Task 23: safe_calculator missing from agent tools"
print(f"✅ Task 23 passed: ReAct agent with {len(react_agent.tools)} tools initialized")

# Task 24
assert agent_result is not None, "Task 24: run_multistep_task returned None"
assert 'output' in agent_result, "Task 24: result dict missing 'output' key"
assert len(agent_result['output']) > 5, "Task 24: output is suspiciously short"
print("✅ Task 24 passed: multi-step agent run completed")

# Task 25
assert aria_agent is not None, "Task 25: aria_agent not created"
assert isinstance(aria_agent, AgentExecutor), f"Task 25: expected AgentExecutor, got {type(aria_agent)}"
agent_prompt_text = aria_agent.agent.llm_chain.prompt.template
assert "ARIA" in agent_prompt_text, "Task 25: ARIA persona not found in agent prompt"
assert aria_result is not None and 'output' in aria_result, "Task 25: aria_agent.invoke failed"
print("✅ Task 25 passed: ARIA persona embedded and agent responds")

print("\n🎉 Section 5 complete!")
print("\n" + "="*60)
print("🏆 ALL 25 TASKS COMPLETE — Lab Finished!")
print("="*60)

---
## 🏁 Lab Complete

**What you built:**

| Section | Skills Demonstrated |
|---|---|
| 1. Foundations | BPE tokenization, softmax, scaled dot-product attention, MoE routing |
| 2. Quantization | Memory math, NF4/BitsAndBytes config, compute dtype selection, VRAM profiling, double quant |
| 3. Inference & Prompting | GGUF loading, temperature/top-p decoding, few-shot, CoT prompting |
| 4. RAG with FAISS | Web scraping, chunking, sentence embeddings, FAISS indexing, RetrievalQA |
| 5. Agents | Tool definition, safe math eval, ReAct initialization, multi-step reasoning, persona tuning |

**Next steps:** Try swapping in larger models (Mistral-7B-Instruct in GGUF), experiment with different chunk sizes in RAG, or extend the agent with a web search tool.